# ColGraphRAG WebQA — Step-by-Step Pipeline Tutorial

This notebook walks through the **query-driven multimodal GraphRAG** pipeline:

| Phase | Script | Role |
|-------|--------|------|
| 0 | `export_webqa_slice.py` *or* prebuilt slice | Build/copy JSONL corpus slice |
| 2 | `pattern.py` | Per-question graph pattern (LLM) |
| 3 | `extraction.py` | Entity/relation extraction (LLM) |
| 4 | `construct.py` | NetworkX → GraphML |
| 5 | `inference.py` | ColEmbed MaxSim retrieval + Gemma answer |
| 6 | `eval/evaluate_webqa_qa.py` | QA-FL / QA-Acc / QA metrics |

**Prerequisites:** Complete the **environment setup** below (venv + `pip install -r requirements.txt`). **Download Hugging Face weights** with `util/download_models.py` (see **Hugging Face model download** later) before real LLM / ColEmbed runs. For real Gemma you also need a CUDA GPU; use `--dry-run` in the pipeline cells for quick tests without loading models.

**Working directory:** Run Jupyter from the repo root `colgraphrag_webqa`, or adjust `REPO` in the **Resolve repo root** cell.

**Two paths:** use **End-to-end run** for a single pass, *or* **Step-by-step phases (manual)** for phase-by-phase execution (do not run both in the same session unless you use different `RUN_ID` / understand the outputs).


## Virtual environment and CUDA dependencies

Do this **once** on your machine before opening this notebook. Commands assume you are in the repo root `colgraphrag_webqa/` (the folder that contains `requirements.txt`, `inference.py`, and `notebook/`).

### Create a venv

**Linux / macOS**

```bash
cd /path/to/colgraphrag_webqa
python3.11 -m venv .venv
source .venv/bin/activate
python -m pip install -U pip setuptools wheel
```

**Windows (PowerShell)**

```powershell
cd C:\path\to\colgraphrag_webqa
py -3.11 -m venv .venv
.\.venv\Scripts\Activate.ps1
python -m pip install -U pip setuptools wheel
```

Use **Python 3.10+** (3.11 recommended).

### Point this notebook at `.venv` (VS Code / Cursor)

After creating the venv, select it as the runtime for notebook cells:

1. Open this `.ipynb` file.
2. Click **Select Kernel** in the top-right of the notebook (or run **Notebook: Select Notebook Kernel** from the Command Palette).
3. Choose **Python Environments** (you may need **Select Another Kernel…** first).
4. Pick **Existing Python environment** and select the interpreter under the repo:  
   **`.venv/bin/python`** (Linux/macOS) or **`.venv\Scripts\python.exe`** (Windows).  
   If `.venv` does not appear, use **Enter interpreter path…** and browse to that path.

If you register an ipykernel display name later (**Jupyter kernel** below), you can select that kernel instead of browsing to `.venv`.

### Install CUDA PyTorch inside the venv (explicit)

With the venv **activated** and `pip` upgraded, install the CUDA 12.4 wheels first (same pins as `requirements.txt`):

```bash
pip install torch==2.6.0+cu124 torchvision==0.21.0+cu124 torchaudio==2.6.0+cu124 \
  --index-url https://download.pytorch.org/whl/cu124 \
  --extra-index-url https://pypi.org/simple
```

Then install the rest of the project (networkx, transformers, etc.):

```bash
pip install -r requirements.txt
```

`pip` will keep the CUDA PyTorch you installed if versions match; otherwise prefer a single pass below.

### Install GPU stack from `requirements.txt` (single command)

The file pins **PyTorch with CUDA 12.4** via `--index-url https://download.pytorch.org/whl/cu124` at the top. Installing the whole file in one go is equivalent to the two-step flow above:

```bash
pip install -r requirements.txt
```

After install, verify CUDA is visible to PyTorch:

```bash
python -c "import torch; print(torch.__version__, torch.cuda.is_available())"
```

You should see a `+cu124` build and `True` when an NVIDIA driver + GPU are present.

### Jupyter kernel (optional)

To run **this** notebook inside the venv:

```bash
pip install ipykernel
python -m ipykernel install --user --name colgraphrag-webqa --display-name "Python (colgraphrag_webqa)"
```

Then select the **Python (colgraphrag-webqa)** kernel in VS Code / Cursor / Jupyter.

**CPU-only note:** `requirements.txt` targets CUDA. For a CPU-only smoke test you would need a separate pin (not covered here); pipeline sections below support `--dry-run` without loading Gemma on GPU.

---


## Available GPU

Run this after activating the venv and installing PyTorch (`requirements.txt`). Shows **NVIDIA driver/GPU** via `nvidia-smi` when available, then **PyTorch CUDA** status via `torch` (skips torch lines if not installed yet).

In [ ]:
import shutil
import subprocess

nv = shutil.which("nvidia-smi")
if nv:
    r = subprocess.run(
        [
            nv,
            "--query-gpu=index,name,driver_version,memory.total,memory.used,memory.free",
            "--format=csv,noheader",
        ],
        capture_output=True,
        text=True,
        timeout=15,
    )
    if r.returncode == 0:
        print("nvidia-smi (GPU list):\n")
        print(r.stdout.strip() or "(empty)")
    else:
        print("nvidia-smi failed:", r.stderr)
else:
    print("nvidia-smi not found in PATH — no NVIDIA CLI, or not a GPU machine.")

print()

try:
    import torch
except ImportError:
    print("torch: not installed yet — complete pip install -r requirements.txt first.")
else:
    print("torch:", torch.__version__)
    print("torch.cuda.is_available():", torch.cuda.is_available())
    if torch.cuda.is_available():
        n = torch.cuda.device_count()
        print("torch.cuda.device_count():", n)
        for i in range(n):
            print(f"  [{i}]", torch.cuda.get_device_name(i))
            print("      mem allocated MB:", torch.cuda.memory_allocated(i) / 1e6)

## Resolve repo root and paths

The pipeline expects `PYTHONPATH` to include the repo root so imports like `mllm`, `util`, `prompt` resolve.

In [ ]:
import os
import sys
from pathlib import Path

# Prefer cwd if it looks like the repo root; else parent of this notebook
_here = Path.cwd()
if (_here / "inference.py").is_file():
    REPO = _here
elif (_here.parent / "inference.py").is_file():
    REPO = _here.parent
else:
    REPO = Path("/workspace/mmgraphrag/lecture/code/colgraphrag_webqa")

assert (REPO / "inference.py").is_file(), f"Set REPO to colgraphrag_webqa root; got {REPO}"

os.chdir(REPO)
sys.path.insert(0, str(REPO))

print("REPO =", REPO.resolve())
print("cwd =", Path.cwd())

## Environment check (optional)

For **real** inference, use `GEMMA4_E4B_IT_TORCH_DTYPE=bf16` on typical single-GPU setups to reduce VRAM.

In [ ]:
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

## Configuration files

- `config/data.yaml` — dataset paths (`WEBQA_DATA_ROOT`, slice locations).
- `config/model.yaml` — Gemma, ColEmbed, fluency model paths and HF repo IDs.
- Copy `.env.example` → `.env` for `HF_TOKEN` if needed.

Models are loaded **repo-local first** (`models/mllm/`, `models/retriever/`) then `/workspace/models/` fallbacks.

In [ ]:
from pathlib import Path

for p in [REPO / "config/data.yaml", REPO / "config/model.yaml"]:
    print(p.name, "exists:" if p.is_file() else "MISSING:", p)

## Hugging Face model download

Weights are **not** included in the git repo. Use `util/download_models.py` to pull checkpoints listed in `config/model.yaml` into `models/` (Gemma, ColEmbed, optional fluency BART for metrics).

- Put **`HF_TOKEN`** or **`HUGGING_FACE_HUB_TOKEN`** in a repo-root **`.env`** if a model is gated (Gemma).
- **Layout after download:** `models/mllm/gemma-4-E4B-it/`, `models/retriever/llama-nemotron-colembed-vl-3b-v2/`, `models/eval/bart-large-cnn/`.
- **CLI:** `python util/download_models.py` (all), or `--only gemma colembed`, or `--dry-run` to print planned paths only.

For model names, paths, and Hugging Face setup, see the root **`README.md`**.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Requires REPO from "Resolve repo root" above; fallback for quick runs
try:
    _r = REPO
except NameError:
    _here = Path.cwd()
    _r = _here if (_here / "util" / "download_models.py").is_file() else _here.parent

dl = _r / "util" / "download_models.py"
if not dl.is_file():
    raise FileNotFoundError(f"Missing {dl}")

# List what would be downloaded (no network files written)
print("--- dry-run ---\n")
subprocess.run([sys.executable, str(dl), "--dry-run"], cwd=str(_r), check=False)

# Uncomment to download (may take a long time and need HF token in .env):
# subprocess.run(
#     [sys.executable, str(dl), "--only", "gemma", "colembed"],
#     cwd=str(_r),
#     env={**os.environ},
#     check=False,
# )

## Toy dataset (shard 14)

If `data/webqa/webqa_shard14_toy/webqa_slice/` is already in the repo, **skip** rebuilding.

Otherwise generate the toy slice (requires full WebQA + baseline files as in `scripts/build_shard14_toy_split.py` defaults):

```bash
python scripts/build_shard14_toy_split.py
```

PNG files may live under `data/webqa/WebQA_imgs_7z_chunks/imgs/all_png/shard_00014/` (or the paths recorded in `webqa_images.jsonl`). Run **Preview toy images** below to display a few samples in the notebook.

In [ ]:
TOY_SLICE = REPO / "data" / "webqa" / "webqa_shard14_toy" / "webqa_slice"
print("Toy slice:", TOY_SLICE)
print("Has webqa_questions.jsonl:", (TOY_SLICE / "webqa_questions.jsonl").is_file())

### Preview toy images (optional)

Reads `webqa_images.jsonl` and displays a few PNGs. Paths in the JSONL may point at `/workspace/...`; if missing, the code tries the repo-local shard folder `data/webqa/WebQA_imgs_7z_chunks/imgs/all_png/shard_00014/` using the same filename.

In [ ]:
import json
from pathlib import Path

from IPython.display import Image as IPyImage, display

try:
    _slice = TOY_SLICE
except NameError:
    try:
        _slice = REPO / "data" / "webqa" / "webqa_shard14_toy" / "webqa_slice"
    except NameError:
        _here = Path.cwd()
        _root = _here if (_here / "data" / "webqa").is_dir() else _here.parent
        _slice = _root / "data" / "webqa" / "webqa_shard14_toy" / "webqa_slice"

try:
    _repo = REPO
except NameError:
    _here = Path.cwd()
    _repo = _here if (_here / "inference.py").is_file() else _here.parent

imgs_jsonl = _slice / "webqa_images.jsonl"
local_shard = _repo / "data" / "webqa" / "WebQA_imgs_7z_chunks" / "imgs" / "all_png" / "shard_00014"

if not imgs_jsonl.is_file():
    print("Missing:", imgs_jsonl)
else:
    rows = []
    with imgs_jsonl.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    print(f"Images in slice index: {len(rows)}")
    for obj in rows[:5]:
        url = obj.get("url") or ""
        p = Path(url)
        if not p.is_file() and p.name:
            alt = local_shard / p.name
            if alt.is_file():
                p = alt
        if not p.is_file():
            print("Skip (file not found):", url[:80], "...")
            continue
        print(obj.get("image_id"), obj.get("title", "")[:60])
        display(IPyImage(filename=str(p), width=400))

## End-to-end run (recommended for learners)

`tests/test_pipeline.py` orchestrates phases 0→6 with the correct env vars. This matches what you run from the terminal.

**Skip this section** if you are using **Step-by-step phases (manual)** (step-by-step cells) instead.

- **`-n 5`** — five questions (fast).
- **`--dry-run`** — no GPU LLM; placeholder answers (good for wiring checks).
- **`GEMMA4_E4B_IT_TORCH_DTYPE=bf16`** — recommended for real runs on limited VRAM.

In [ ]:
import os
import subprocess

# Toggle: set True for a quick CPU-friendly smoke test
DRY_RUN = True
N_QUERIES = 2

env = os.environ.copy()
env["PYTHONPATH"] = str(REPO)
if not DRY_RUN:
    env.setdefault("GEMMA4_E4B_IT_TORCH_DTYPE", "bf16")

cmd = [
    sys.executable,
    str(REPO / "tests" / "test_pipeline.py"),
    "-n", str(N_QUERIES),
]
if DRY_RUN:
    cmd.append("--dry-run")

print(" ".join(cmd))
r = subprocess.run(cmd, cwd=str(REPO), env=env)
print("exit:", r.returncode)

After a successful run, outputs live under **`result/<RUN_ID>/`**:

- `webqa_slice/` — copied or exported JSONL
- `phase2_pattern_cache/`, `phase3_extraction_cache/`
- `phase4_graphs_out/*.graphml`
- `phase5_inference/predictions.json`, `evaluation_report.json`

In [ ]:
# List latest result runs
result_root = REPO / "result"
if result_root.is_dir():
    runs = sorted(result_root.iterdir(), key=lambda p: p.stat().st_mtime, reverse=True)
    for p in runs[:5]:
        print(p.name)
else:
    print("No result/ yet — run the pipeline first.")

## Inspect one GraphML (Phase 4 output)

Graphs are standard NetworkX GraphML — nodes carry `entity_name`, `type`, `description`, `source_id`.

In [ ]:
import networkx as nx

graphs_dir = REPO / "result"
gml_files = list(graphs_dir.glob("**/phase4_graphs_out/*.graphml"))
if not gml_files:
    print("No GraphML found under result/*/phase4_graphs_out/")
else:
    path = sorted(gml_files)[0]
    G = nx.read_graphml(path)
    print("File:", path.relative_to(REPO))
    print("Nodes:", G.number_of_nodes(), "Edges:", G.number_of_edges())
    for nid in list(G.nodes)[:3]:
        print(" -", nid, dict(G.nodes[nid]))

## Step-by-step phases (manual)

Set `RUN_ID`, `N_QUERIES`, and `DRY_RUN` in the next cell, then run Phase 0 → 6 in order. This mirrors `tests/test_pipeline.py` (toy dataset, local paths).

**Skip this section** if you already ran **End-to-end run** for a full pass (or use a new `RUN_ID` here).

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

# --- student knobs ---
RUN_ID = "notebook_tutorial_run"  # change per experiment
N_QUERIES = 2
DRY_RUN = True  # False = real Gemma (needs CUDA + models)

# --- paths (same logic as tests/test_pipeline.py) ---
_LOCAL_DATA = REPO / "data" / "webqa"
if (_LOCAL_DATA / "webqa_shard14_toy" / "webqa_slice").is_dir():
    TOY_SLICE = _LOCAL_DATA / "webqa_shard14_toy" / "webqa_slice"
else:
    TOY_SLICE = Path("/workspace/data/webqa/WebQA_imgs_7z_chunks/webqa_shard14_toy/webqa_slice")

if (_LOCAL_DATA / "WebQA_imgs_7z_chunks" / "imgs" / "all_png" / "shard_00014").is_dir():
    SHARD14_IMGS = str(_LOCAL_DATA / "WebQA_imgs_7z_chunks" / "imgs" / "all_png" / "shard_00014")
else:
    SHARD14_IMGS = "/workspace/data/webqa/WebQA_imgs_7z_chunks/imgs/all_png/shard_00014"

result_dir = REPO / "result" / RUN_ID
slice_dir = result_dir / "webqa_slice"
q_file = slice_dir / "webqa_questions.jsonl"
t_file = slice_dir / "webqa_texts.jsonl"

dry_run = "1" if DRY_RUN else "0"

from util.llm_defaults import DEFAULT_GEMMA4_E4B_IT_MODEL_PATH
gemma_path = os.environ.get("GEMMA4_E4B_IT_MODEL_PATH", "").strip() or str(DEFAULT_GEMMA4_E4B_IT_MODEL_PATH)

common_env = {
    "PYTHONPATH": str(REPO),
    "MMGRAPHRAG_RUN_ID": RUN_ID,
    "WEBQA_RUN_PROFILE": "val_n100",
    "WEBQA_DATA_ROOT": str(_LOCAL_DATA) if _LOCAL_DATA.is_dir() else "/workspace/data/webqa",
    "PATTERN_MAX_SAMPLES": str(N_QUERIES),
    "WEBQA_EXPORT_MAX": str(N_QUERIES),
    "EXTRACTION_MAX_QUESTIONS": str(N_QUERIES),
    "CONSTRUCT_MAX_QUESTIONS": str(N_QUERIES),
    "PATTERN_DRY_RUN": dry_run,
    "EXTRACTION_DRY_RUN": dry_run,
    "PATTERN_CONCURRENCY": "1",
    "EXTRACTION_CONCURRENCY": "1",
    "GEMMA4_E4B_IT_MODEL_PATH": gemma_path,
    "VIDORE_TEXT_LLM_BACKEND": "hf_gemma_4_e4b_it",
    "COLEMBED_MODEL_PATH": os.environ.get(
        "COLEMBED_MODEL_PATH",
        "/workspace/models/retriever/llama-nemotron-colembed-vl-3b-v2",
    ),
}
if os.environ.get("GEMMA4_E4B_IT_TORCH_DTYPE"):
    common_env["GEMMA4_E4B_IT_TORCH_DTYPE"] = os.environ["GEMMA4_E4B_IT_TORCH_DTYPE"]
elif not DRY_RUN:
    common_env.setdefault("GEMMA4_E4B_IT_TORCH_DTYPE", "bf16")

def run_step(desc: str, cmd: list[str], env: dict) -> int:
    merged = {**os.environ, **env}
    print(f"\n=== {desc} ===\n{' '.join(cmd)}\n")
    r = subprocess.run(cmd, cwd=str(REPO), env=merged)
    print(f"exit={r.returncode}")
    return r.returncode

print("RUN_ID:", RUN_ID, "| N_QUERIES:", N_QUERIES, "| DRY_RUN:", DRY_RUN)
print("Toy slice:", TOY_SLICE)

### Copy pre-built toy `webqa_slice` into `result/<RUN_ID>/`

In [ ]:
if not TOY_SLICE.is_dir():
    raise FileNotFoundError(f"Toy slice missing: {TOY_SLICE}")

result_dir.mkdir(parents=True, exist_ok=True)
if slice_dir.exists():
    shutil.rmtree(slice_dir)
shutil.copytree(TOY_SLICE, slice_dir)
print("Copied to", slice_dir)

### Pattern (`pattern.py`)

In [ ]:
pattern_cache = result_dir / "phase2_pattern_cache"
_local_pattern = _LOCAL_DATA / "webqa_shard14_toy" / "webqa_slice" / "webqa_questions.jsonl"
pattern_question_file = str(_local_pattern) if _local_pattern.is_file() else "/workspace/data/webqa/WebQA_data_first_release/WebQA_train_val.json"

pattern_env = {
    **common_env,
    "PATTERN_JSON_FILE_PATH": pattern_question_file,
    "PATTERN_CACHE_DIR": str(pattern_cache),
}
run_step("Phase 2 pattern", [sys.executable, "pattern.py"], pattern_env)

### Extraction (`extraction.py`)

In [ ]:
extraction_cache = result_dir / "phase3_extraction_cache"
extraction_env = {
    **common_env,
    "EXTRACTION_QUESTION_FILE": str(q_file),
    "EXTRACTION_TEXT_FILE": str(t_file),
    "EXTRACTION_PATTERN_CACHE_DIR": str(pattern_cache),
    "EXTRACTION_CACHE_DIR": str(extraction_cache),
}
run_step("Phase 3 extraction", [sys.executable, "extraction.py"], extraction_env)

### Construct GraphML (`construct.py`)

In [ ]:
graph_dir = result_dir / "phase4_graphs_out"
construct_env = {
    **common_env,
    "CONSTRUCT_QUESTION_FILE": str(q_file),
    "CONSTRUCT_TABLE_FILE": str(slice_dir / "webqa_tables.jsonl"),
    "CONSTRUCT_IMAGE_FILE": str(slice_dir / "webqa_images.jsonl"),
    "CONSTRUCT_TEXT_FILE": str(t_file),
    "CONSTRUCT_EXTRACTION_CACHE": str(extraction_cache),
    "CONSTRUCT_OUTPUT_GRAPH_DIR": str(graph_dir),
    "CONSTRUCT_WEBQA_SLICE_DIR": str(slice_dir),
}
run_step("Phase 4 construct", [sys.executable, "construct.py"], construct_env)

### Inference (`inference.py`)

Uses ColEmbed retrieval when enabled and Gemma for answers (unless `DRY_RUN`).

In [ ]:
phase5_dir = result_dir / "phase5_inference"
phase5_dir.mkdir(parents=True, exist_ok=True)
pred_json = phase5_dir / "predictions.json"

inference_env = {
    **common_env,
    "MMGRAPHRAG_RUN_ID": RUN_ID,
    "INFERENCE_GRAPH_DIR": str(graph_dir),
    "INFERENCE_QUESTION_FILE": str(q_file),
    "INFERENCE_OUTPUT_JSON": str(pred_json),
    "INFERENCE_MAX_QUESTIONS": str(N_QUERIES),
    "INFERENCE_DRY_RUN": dry_run,
    "INFERENCE_COLEMBED_RETRIEVAL": os.environ.get("INFERENCE_COLEMBED_RETRIEVAL", "1"),
    "INFERENCE_WEBQA_SLICE_DIR": str(slice_dir),
    "WEBQA_IMGS_DIR": SHARD14_IMGS,
}
run_step("Phase 5 inference", [sys.executable, "inference.py"], inference_env)

### QA evaluation (`eval/evaluate_webqa_qa.py`)

In [ ]:
if pred_json.is_file():
    report_json = phase5_dir / "evaluation_report.json"
    eval_env = {**common_env, "MMGRAPHRAG_RUN_ID": RUN_ID}
    run_step(
        "Phase 6 eval",
        [
            sys.executable,
            "eval/evaluate_webqa_qa.py",
            "--predictions",
            str(pred_json),
            "--gold_jsonl",
            str(q_file),
            "--report_json",
            str(report_json),
            "--split_label",
            "val",
        ],
        eval_env,
    )
else:
    print("Skip eval: predictions.json missing")

## Appendix: raw shell commands (optional)

The **Step-by-step phases (manual)** cells are the recommended way to run phases from the notebook. Below are **bash** equivalents for terminal-only workflows. Replace `$RUN_ID`, paths, and limits to match your machine.

**Phase 0 — export slice** (skip if you already copied `webqa_slice` into `result/$RUN_ID/`):
```bash
export MMGRAPHRAG_RUN_ID=my_run WEBQA_EXPORT_MAX=5 WEBQA_SLICE_DIR=result/my_run/webqa_slice
python export_webqa_slice.py
```

**Phase 2 — pattern:**
```bash
export PATTERN_MAX_SAMPLES=5 PATTERN_CACHE_DIR=result/my_run/phase2_pattern_cache
python pattern.py
```

**Phase 3 — extraction:**
```bash
export EXTRACTION_QUESTION_FILE=result/my_run/webqa_slice/webqa_questions.jsonl ...
python extraction.py
```

Full env wiring is identical to `tests/test_pipeline.py` — copy from there for reproducibility.

## Demo UI (optional)

After you have a `result/<RUN_ID>/` with predictions and graphs, configure `demo/be/config/paths.yaml` (often `run_id: latest`). See `demo/README.md`.

You can either use **terminal commands** or the **Start demo BE + FE** cell below to start servers from this notebook (background `subprocess`). Requirements:

- **BE:** same Python venv as the pipeline; ports **8000** (override with env `DEMO_BE_PORT`).
- **FE:** **Node.js + npm** (e.g. LTS via nvm); port **5173** (`DEMO_FE_PORT`). Vite proxies `/api` to the backend.

Open in the browser: **http://127.0.0.1:5173/** (FE) — API health: **http://127.0.0.1:8000/health**.

### Terminal equivalent

```bash
cd demo/be && python server.py --host 0.0.0.0 --port 8000
cd demo/fe && npm install && npm run dev
```

### Start demo BE + FE from the notebook

Runs servers as **background processes** so you can keep using the notebook. Run **Resolve repo root and paths** first so `REPO` is defined (otherwise the cell falls back to the current working directory).

Re-running this cell terminates any servers started earlier by this notebook session (`DEMO_PROCESSES`).

In [ ]:
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

try:
    _repo = REPO
except NameError:
    _here = Path.cwd()
    _repo = _here if (_here / "demo" / "be" / "server.py").is_file() else _here.parent

be_dir = _repo / "demo" / "be"
fe_dir = _repo / "demo" / "fe"
if not (be_dir / "server.py").is_file():
    raise FileNotFoundError(f"Expected demo backend at {be_dir}")

if "DEMO_PROCESSES" not in globals():
    DEMO_PROCESSES = []
else:
    for _p in list(DEMO_PROCESSES):
        if _p.poll() is None:
            _p.terminate()
    DEMO_PROCESSES.clear()

BE_PORT = int(os.environ.get("DEMO_BE_PORT", "8000"))
FE_PORT = int(os.environ.get("DEMO_FE_PORT", "5173"))

be_env = {**os.environ, "PYTHONPATH": str(be_dir)}
p_be = subprocess.Popen(
    [sys.executable, "server.py", "--host", "0.0.0.0", "--port", str(BE_PORT)],
    cwd=str(be_dir),
    env=be_env,
    stdin=subprocess.DEVNULL,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    start_new_session=True,
)
DEMO_PROCESSES.append(p_be)
print(f"Demo BE started | pid={p_be.pid} | http://127.0.0.1:{BE_PORT}/health")
time.sleep(1.5)

npm = shutil.which("npm")
if npm is None:
    print("npm not in PATH — install Node.js, then: cd demo/fe && npm install && npm run dev")
else:
    if not (fe_dir / "node_modules").is_dir():
        print("Running npm install in demo/fe (first time may take a minute)...")
        subprocess.run([npm, "install"], cwd=str(fe_dir), check=False)
    p_fe = subprocess.Popen(
        [npm, "run", "dev"],
        cwd=str(fe_dir),
        env={**os.environ},
        stdin=subprocess.DEVNULL,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        start_new_session=True,
    )
    DEMO_PROCESSES.append(p_fe)
    print(f"Demo FE started | pid={p_fe.pid} | http://127.0.0.1:{FE_PORT}/")

print("\nOpen the frontend URL above. Stop servers with the next **Stop demo servers** cell.")

### Stop demo servers

Terminates processes recorded in `DEMO_PROCESSES` from **Start demo BE + FE**. If something is still listening on 8000/5173, stop it manually in a terminal (`kill`, or Task Manager on Windows).

In [ ]:
import time as _time

if "DEMO_PROCESSES" in globals() and DEMO_PROCESSES:
    for _p in DEMO_PROCESSES:
        if _p.poll() is None:
            _p.terminate()
    _time.sleep(0.5)
    for _p in DEMO_PROCESSES:
        if _p.poll() is None:
            _p.kill()
    DEMO_PROCESSES.clear()
    print("Demo BE/FE processes terminated.")
else:
    print("No DEMO_PROCESSES to stop (run Start demo BE + FE first).")

## Further reading

- `README.md` — overview, data layout, model download, and pipeline overview.
- `tests/test_pipeline.py` — full orchestration and environment variables for each phase.